### Prompts

In [1]:
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import (StrOutputParser, 
                        CommaSeparatedListOutputParser, JsonOutputParser, PydanticOutputParser)
from langchain_core.runnables import RunnableLambda, RunnableParallel, RunnablePassthrough, RunnableGenerator, RunnableAssign
import os
from dotenv import load_dotenv

load_dotenv()
os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY")

In [2]:
from langchain_core.prompts import (ChatPromptTemplate,  PromptTemplate, 
                                    ChatMessagePromptTemplate, HumanMessagePromptTemplate, SystemMessagePromptTemplate)
from langchain_core.output_parsers import JsonOutputParser
import json
from langchain_core.messages import HumanMessage, SystemMessage

# Initialize Groq LLM
llm = ChatOpenAI(
    model="gpt-4o",
    temperature=0.7
)
print(llm)

profile={'name': 'GPT-4o', 'release_date': '2024-05-13', 'last_updated': '2024-08-06', 'open_weights': False, 'max_input_tokens': 128000, 'max_output_tokens': 16384, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'pdf_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True} client=<openai.resources.chat.completions.completions.Completions object at 0x0000014D54093AD0> async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x0000014D55581A10> root_client=<openai.OpenAI object at 0x0000014D540935D0> root_async_client=<openai.AsyncOpenAI object at 0x0000014D55581590> model_name='gpt-4o' temperature=0.7 model_kwargs={} openai_api_key=SecretStr('*******

In [3]:
model = llm

In [5]:
from langchain_core.messages import HumanMessage,SystemMessage

# A manual way of creating messages
messages=[
    SystemMessage(content="Translate the following from English to French"),
    HumanMessage(content="Hello How are you?")
]
print(messages)

result=model.invoke(messages)
print(result)

[SystemMessage(content='Translate the following from English to French', additional_kwargs={}, response_metadata={}), HumanMessage(content='Hello How are you?', additional_kwargs={}, response_metadata={})]
content='Bonjour, comment ça va ?' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 6, 'prompt_tokens': 23, 'total_tokens': 29, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_88aa8039ec', 'id': 'chatcmpl-DXL98DLcPxUM44goybfDd2fTQs4dE', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019db3d3-93fe-7de2-9bd6-2fdddf917a7f-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 23, 'output_tokens': 6, 'total_tokens': 29, 'input_token_details': {'audio': 

In [8]:
messages = [
    SystemMessage(content="Translate the following from English to {language}"),
    HumanMessage(content="{text}")
]
## Cant work

In [7]:
from langchain_core.messages import SystemMessage, HumanMessage

language = "French"
text = "Hello"

messages = [
    SystemMessage(content=f"Translate the following from English to {language}"),
    HumanMessage(content=text)
]
print(messages)

[SystemMessage(content='Translate the following from English to French', additional_kwargs={}, response_metadata={}), HumanMessage(content='Hello', additional_kwargs={}, response_metadata={})]


In [11]:
# solution 2: To use HumanMessagePromptTemplate and its counterpart
human = HumanMessagePromptTemplate.from_template("{input}")
system = SystemMessagePromptTemplate.from_template(
    "Translate the word from English to {language}"
)

prompt = ChatPromptTemplate.from_messages([system, human])

result = prompt.invoke({
    "input": "Hello, are you doing today",
    "language": "French"
})

print(result)

messages=[SystemMessage(content='Translate the word from English to French', additional_kwargs={}, response_metadata={}), HumanMessage(content='Hello, are you doing today', additional_kwargs={}, response_metadata={})]


In [12]:
chain = prompt | model 
chain.invoke({"input": "Hello, how are you doing today", "language": "Arabic"})

AIMessage(content='مرحباً، كيف حالك اليوم؟', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 25, 'total_tokens': 34, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_34ca2faac5', 'id': 'chatcmpl-DXLJqg7UFsmETLBKqYU0uqevXtmYT', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019db3dd-b3b7-7c93-9a0c-f48e6789f117-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 25, 'output_tokens': 9, 'total_tokens': 34, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [13]:
chain.invoke({"input": "Hello, how are you doing today", "language": "Arabic"}).content

'مرحباً، كيف حالك اليوم؟'

In [15]:
template = ChatPromptTemplate(
            [
                ("system", "Translate the word from Eglish to {language}."),
                ("human", "{input}"),
            ]
        )
template.invoke({
    "language": "Arabic",
    "input": "Hello, how are you doing, and how is family and work?"
})

ChatPromptValue(messages=[SystemMessage(content='Translate the word from Eglish to Arabic.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Hello, how are you doing, and how is family and work?', additional_kwargs={}, response_metadata={})])

In [16]:
chain = template | model

In [17]:
chain.invoke(
    {
    "language": "Arabic",
    "input": "Hello, how are you doing, and how is family and work?"
}
)

AIMessage(content='مرحبًا، كيف حالك، وكيف حال العائلة والعمل؟', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 35, 'total_tokens': 49, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_07a5e8f420', 'id': 'chatcmpl-DXLNiBox80LfP7TyaMn03YPGuZj3X', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019db3e1-5df1-7ff2-8a20-4b6c0eb2bfe8-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 35, 'output_tokens': 14, 'total_tokens': 49, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [18]:
chain.invoke(
    {
    "language": "Arabic",
    "input": "Hello, how are you doing, and how is family and work?"
}
).content

'مرحبًا، كيف حالك، وكيف حال العائلة والعمل؟'

In [19]:
chain.invoke(
    {
    "language": "Arabic",
    "input": "Hello, how are you doing, and how is family and work? It's great seeing you, I hope you are also"
}
).content

'مرحبًا، كيف حالك، وكيف حال العائلة والعمل؟ إنه من الرائع رؤيتك، آمل أن تكون أيضًا'

In [20]:
template = ChatPromptTemplate(
            [
                ("system", "Write me very wonderful poem in {language}."),
                ("human", "{input}"),
            ]
        )
template.invoke({
    "language": "Arabic",
    "input": "Life"
})

ChatPromptValue(messages=[SystemMessage(content='Write me very wonderful poem in Arabic.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Life', additional_kwargs={}, response_metadata={})])

In [21]:
chain = template | model

In [23]:
print(chain.invoke({
    "language": "Arabic",
    "input": "Life"
}).content)

في بحر الحياة نمضي بلا انتهاء  
تأخذنا الأمواج في كل اتجاه  
تنسج لنا الأيام قصصًا وأحلامًا  
وما بين الفرح والحزن نعيش الأمل  

كل صباح شمس تشرق من جديد  
تُضيء لنا الدروب وتفتح الأبواب  
نرسم بالألوان لوحات الأماني  
ونزرع الأمل في قلوب الأحباب  

وفي الليل تحت ضوء القمر  
نحكي للنجوم أسرارنا وهمومنا  
نسافر في خيالنا إلى عوالم بعيدة  
ونحلم بغدٍ مشرق وسعيد  

الحياة رحلة مليئة بالتجارب  
تعلمنا كيف نكون أقوى وأعمق  
نواجه الصعاب بابتسامة وثبات  
ونمضي في دروبنا بلا استسلام  

فكن للحياة عاشقًا ومتفائلًا  
وأطلق العنان لأحلامك وأمالك  
واعلم أن في كل لحظة جمالًا  
وفي كل نفس فرصة للعيش وللحلم  


In [24]:
template = ChatPromptTemplate(
            [
                ("system", "Write me very wonderful poem in {language}. After go immediately after completing the poem and then transalate into {language2}"),
                ("human", "{input}"),
            ]
        )
template.invoke({
    "language": "Arabic",
    "language2": "English",
    "input": "Life"
})

ChatPromptValue(messages=[SystemMessage(content='Write me very wonderful poem in Arabic. After go immediately after completing the poem and then transalate into English', additional_kwargs={}, response_metadata={}), HumanMessage(content='Life', additional_kwargs={}, response_metadata={})])

In [25]:
print(chain.invoke({
    "language": "Arabic",
    "language2": "English",
    "input": "Life"
}).content)

الحياةُ بحرٌ واسعُ الأرجاءِ  
تُداعبُنا الأمواجُ في كلِّ مساءِ  

تارةً نرتفعُ نحو السماءِ  
وتارةً ننخفضُ نحوَ الأعماقِ الخفيّةِ  

هي شمسٌ تُشرقُ بعدَ الظلامِ  
تُضيءُ لنا الدروبَ بالآمالِ الوضّاءةِ  

هي زهرةٌ تتفتحُ في الربيعِ  
تُعطّرُ الأجواءَ بالحبِّ والعطاءِ  

هي لحظةُ فرحٍ لا تُنسى  
وذكرى حزنٍ نتعلمُ منها الدروسَ  

نحياها بحلوها ومرّها  
ونكتبُ فيها فصولَ قصصنا الفريدةِ  

فالحياةُ ليست سوى رحلةٍ  
نبحرُ فيها بينَ الأملِ والعناءِ  

فلنعيشها بكلِّ شغفٍ وحبٍّ  
ونزرعُ فيها الخيرَ والوفاءَ بلا انتهاءِ  


In [26]:
template = ChatPromptTemplate.from_messages([
    ("system", 
     "Write a beautiful poem in {language} about the given topic. "
     "Then provide a translation of the poem in {language2}. "
     "Format your response clearly with two sections:\n\n"
     "Poem ({language}):\n...\n\n"
     "Translation ({language2}):\n..."
    ),
    ("human", "{input}")
])

chain = template | model 

print(chain.invoke({
    "language": "Arabic",
    "language2": "English",
    "input": "Life"
}).content)

Poem (Arabic):
الحياةُ بستانٌ من الألوانِ،  
تُزهرُ فيهِ كلُّ الأمانِ،  
أيامٌ تمرُّ كنسمةِ الريحِ،  
وأخرى كصوتِ الألحانِ.

تُعلِّمُنا الفرحَ في الصباحِ،  
وترسمُ الحزنَ في ثنايا المساءِ،  
ولكننا نجدُ فيها الضياء،  
في كل زاويةٍ من الأرجاءِ.

نجوبُ دروبَها بحلمٍ كبيرٍ،  
ونسعى نحوَ نورٍ منيرٍ،  
نُصارعُ الأحزانَ والأفراحَ،  
ونبني في القلبِ قصورَ الأملِ.

فالحياةُ هديةٌ من السماءِ،  
نعيشُها بكلِّ حبٍ ووفاءِ،  
نزرعُ فيها بذورَ الخيرِ،  
ونرويها من نهرِ العطاءِ.

Translation (English):
Life is a garden of colors,  
Where every hope blossoms,  
Days pass like a breeze,  
And others like the sound of melodies.

It teaches us joy in the morning,  
And paints sorrow in the folds of the evening,  
Yet we find light within it,  
In every corner of the realms.

We journey through its paths with a big dream,  
And strive towards a shining light,  
We wrestle with sorrows and joys,  
And build castles of hope in the heart.

For life is a gift from the heavens,  
We live it with all love and l